<a href="https://colab.research.google.com/github/davidsmall-Durham/EDS-Practicals/blob/main/EDS_Practical_5_Part_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environmental Data Science Practical 4: Part 2

---




This is the main part of this practical. As well as expanding on what you covered in the previous notebook we will also explore an alternative way of storing data when working with Google Colab.

The main thrust of this part of the practical is to get you performing (relatively) simple data extraction and analyses on some NetCDF data.  Specifically, we are going to plot a temperature time-series derived from a 3-D data set and we are going to calculate a simple temperature anomaly for the summer of 2003 which was exceptionally warm in Western Europe. You can read more about the 2003 heatwave from a [casestudy by the UK MetOffice](https://weather.metoffice.gov.uk/learn-about/weather/case-studies/heatwave).

A temperature anomaly is simply the difference between a measured temperature and a long-term average (baseline) for a specific location/area and time.

### **Aims**
The aims of this part of the practical are...

*   Learn how to link our Google Drive to Colab and download and work with files stored in it.
*   Introduce you to the Copernicus Climate Data Store (CDS) and the Python Application Programming Interface (API) used to access it's datasets.
*   Perform a simple calcualtions and analyses on multi-dimensional data.

By the end of this practical you should be able to mount you Google Drive and download files to a user defined folder. Be familiar with the Copernicus Climate Data Store and how we can download data directly using Python.

---

### Working through the practical
As with the previous notebook, firstly:

1. **COPY THE NOTEBOOK TO YOUR GOOGLE DRIVE** using the "Copy to Drive" button at the top of the page.
2. Read through the instructions and execute each code block cell to see what it does. If you can't see codeblocks click on the arrow next to the Heading.
3. Answer the questions in each section and tackle the tasks that require you to write (very short) bits of code.


## **Setup**



As before we need to install required packages and import them into the local colab environment.

We will install the same packages as before but becasue we are using the CDS we also need to install the approapriate Application Programming Interface (API) package `cdsapi`. See [here](https://cds.climate.copernicus.eu/how-to-api) for more information.

In [ ]:
!pip install xarray -q cartopy -q netcdf4 -q cdsapi -q

As before we need to import the packages into the Colab environment.

In [ ]:
# Copernicus Climate Data Store API
import cdsapi

# Packages for handling file paths
from pathlib import Path
import os

# Libraries for working with multidimensional arrays
import numpy as np
import xarray as xr

# Libraries for plotting and visualising data
import matplotlib.path as mpath
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.feature as cfeature


# Disable warnings for data download via API
import urllib3
urllib3.disable_warnings()

## **Section 1: Mount Google Drive and set the path**


In the last notebook we stored our downloaded file directly in Colab.

This time we use our Google Drive so we need to do import and mount our Drive.


In [ ]:
#@title ##### This cell mounts your Google Drive
from google.colab import drive
drive.mount('/content/drive')

We now set the directory path in our Google Drive. This is where we will download and save files to.

We give it a variable name (`DATADIR`).

If you want to use a folder named 'ESD_Practical_4' in your 'My Drive' folder your code would read:

```
DATADIR = '/content/drive/My_Drive/ESD_Practical_4'
```

*N.B. you don't need to make this folder first!*

In [ ]:
#@title ##### Define the path to your folder for this practical - call it what you like but make sure it is informative


### **Questions**

1. Looking at the file explorer on the left of the screen, why can't you see a folder with that name in your Google Drive?
2. What are the advantages of mounting your Google Drive and working from it?



## **Section 2: Download the data**

We will request data from the Climate Data Store (CDS) programmatically using the CDS Application Programming Interface (API).

First, we need to define two variables: `URL` and `KEY` which are used to build the CDS API key (our unique identifier).

Your KEY includes your personal User ID and CDS API key. To obtain these visit https://cds.climate.copernicus.eu/how-to-api.


In [ ]:
#@title ##### Set the URL and  for the CDS and add your key

# These are copied and pasted from the API page
URL = 'https://cds.climate.copernicus.eu/api'

KEY=''


In this part of the practical we will be working with ERA5 climate reanalysis data (https://climate.copernicus.eu/climate-reanalysis).

As we are looking at a heatwave we will download the variable `2m_temperature`, the air temperature as measured 2 metres above ground level.

ERA5 provides hourly, daily, and monthly products available. We will download monthly data.

To download the data we need some code that is provided by the [CDS website](https://cds.climate.copernicus.eu/datasets).

The API request code takes the format:

```
dataset = "name of dataset"
request = {
    "A parameter name": ["value that correspond to chosen options"],
    "Next parameter": ["value"
    "Another parameter": ["first value", second value", "and so on"]
    }

```

We then set the download request running with the lines:

```
client = cdsapi.Client(url=URL, key=KEY) # our credentials to allow the download
client.retrieve(dataset, request).download(full_path) # tells the server what to deliver (dataset, request) and the path to our download location including the filename
```


Go to [this page](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means?tab=download) and get the API code for **monthly averaged reanlalysis** of **2 m temperature** for the months of **June, July and August** in the years **1940-2025**. change the extent to `[55, -10, 36, 10] ` (e.g., Western Europe) and set the file format to **NetCDF4** and **unarchived**.

N.B extent has the format `[North, West, South, East]`

Once you have selected the correct options **copy and paste** your the generated code into the code cell below as indicated.

In [ ]:
#@title ##### This code cell makes the folder and downloads the file

# Create the folder
Path(DATADIR).mkdir(parents=True, exist_ok=True)

# Define an informative filename for the file to be downloaded
filename = "era5_W_Europe_summer_temps_2m.nc"

# Now define the full path name (the address for the file). It joins the DATADIR variable with the filename variable into a single string of text
full_path = os.path.join(DATADIR, filename)


# First we check if the file already exists (e.g., if we run this notebook
# again there is no point waiting to donwload the file again!).
if os.path.exists(full_path):
    print("File already exists! Skipping the CDS download entirely.")
else:
    print("File not found. Connecting to CDS to download...")

    # Now we can add our API Request code below




    # Pass the URL and KEY variables to the Client constructor
    client = cdsapi.Client(url=URL, key=KEY)
    client.retrieve(dataset, request).download(full_path) # !!! This is the line of code that actually instructs Python to download the data to the designated path location

### **Questions**

1. How many months of data have you downloaded?
2. What months of the year have you downloaded?
3. What range of *latitude* does our study area include?

## **Section 3: Open and inspect the downloaded data**

We have downloaded monthly temperatures for the summer months (June, July, August) for an area covering Western Europe.

As before we load the file using `xarray` (`xr`) and inspect it using `print`.



In [ ]:
#@title ##### Open the file and examine the file structure by printing the dataset


### **Questions**
1. What is the spatial resolution of the data (i.e. how many grid cells)?
2. How many individual data points are there?
3. What are the units of the temperature data?



**Convert dataset to data array**

In `xarray` a dataset is a specific type of object with certain properties. It can have multiple variables (as in part 1) but it can  be awkward to work with (especially if its very big).

We can convert the variable we are interested in `t2m` into a data array (a simpler type of variable). It still has co-ordinate data associated with it (and thus so will any other arrays we make from it).

When we do this we will also convert our units from Kelvin to Celsius

In [ ]:
#@title ##### Convert to data array and change units

# Convert to celsius

When we are working with datasets like these it is usually because we want to do some sort of analysis on them.

Most analyses require us to  **slice** and/or **aggregate** the data across its dimensions.

There are numerous approaches and it depends on the questions we are interested in. For example (non-exhaustive)

| Analysis | Operation | Output Dimensions | Best Visualisation |
| :--- | :--- | :--- | :--- |
| **Instantaneous Snapshot**<br>*(e.g., "What was the temperature during the July 2023 heatwave?")* | **Slice** `time` | 2D (`lat`, `lon`) | 2D Spatial Map / Raster |
| **Whole-region (1D) timeseries**<br>*(e.g., "Is warming accelerating in the region?")* | **Aggregate** (`lat`, `lon`) via `.mean()` | 1D (`time`) | 1D Line Graph (Time Series) |
| **Long-term Climate Baseline**<br>*(e.g., "Where are the warmest areas on average during summer?")* | **Aggregate** `time` via `.mean()` | 2D (`lat`, `lon`) | 2D Spatial Map |
| **Point-specific (1D) timeseries**<br>*(e.g., "How has temperature varied at London's coordinates?")* | **Slice** `lat` & `lon` | 1D (`time`) | 1D Line Graph |
| **Latitudinal Zonal Profile**<br>*(e.g., "How does temperature change from North to South?")* | **Aggregate** `lon` & `time` | 1D (`lat`) | 1D Transect / Profile Line |




### **Questions**

Try to visualise what this data set (or datacube ) "looks" like. Think about the dimensions as well as the number of data points and what they represent.

1.   Can you write a brief 'plain language' description of the structure of this datacube?

2.  What combination of **slicing/aggregating** would we undertake to calculate the average summer temperatures in Durham since the year 2000?

## **Section 4: Plotting a 1-D timeseries and fitting a trendline**

Summer temperatures (northern hemisphere) are defined as the average of June/July/August (JJA). Thus to find the year with the warmest summer across our entire area we need to:

1.  Calculate the average the JJA temperatures for every grid cell in each year.
2.  Weight the data by latitude (don't worry about the maths behind this!)
3.  Calculate an overall average for each year across the whole Area of Interest (AOI).

---

#####  *Info on weighting*
If our data covers a large area (like a continent), a simple `.mean()` on `latitude` and `longitude` is inaccurate because grid cells defined by latitude and longitude get smaller as you move toward the poles.

To get a "true" average, we should weight by latitude.

In [ ]:
# @title #### This codecell calculates a spatial average for every year

#  Group our data by year and calculate the mean across the time dimension
#  ('valid_time').


# This is an example of daisy chaining
# operations. First we grouped the values by year, then we calculated the mean of each year.


# Calculate our weightings


# First we weight the data, then we calculate the spatial mean.


# Find the year with the maximum value


print(f"The warmest summer was {warmest_year} with an average of {warmest_val:.2f}")

The warmest summer was 2003 with an average of 21.15


We have now got a 1-D timeseries of summer temperature in Western Europe for the period 1940 - 2025. We can plot that up to see what it looks like.

We use the `.plot` function on our desired variable `yearly_avg` and then compute a trendline using the `polyfit` function in `numpy`. We are using the standard linear equation ($y = mx + b$). We make a variable `z` which stores the slope ($m$) and intercept ($b$) of the trendline.

The `.poly1d` command builds a predictor function `p` which takes the numbers stored in our `z` variable and turns them into a useable format for plotting. We have to tell Python to `.plot` this trend line. We can now also use `p` to predict the temperature for any year. e.g., `p(2050)` would return the predicted temperature in 2050.

We will save this figure to our Google Drive using the `savefig` command. We need to tell python what to call the file and where to save it (the path).


In [ ]:
# @title ##### This codecell plots the data and fits a trendline
# Set up the figure
plt.figure(figsize=(12, 6))

# Plot the weighted summer means and set the line type etc
# xarray's .plot() wrapper automatically handles the time/year axis
yearly_avg.plot(marker='o', markersize=4, linestyle='-', color='#d62728', label='ADD YOUR LABEL')

# We add a trendline here. The funtion needs the x and y coordinates of the
# points being fitted. In our case this is the year and the weighted temperature
# values of the yearly_avg variable. The '1' indicates we are fitting a simple
# first degree polynomial (i.e. a straight line). This helps visualize the
# long-term warming trend.


# Get the slope of the fitted line (e.g., how fast is it warming)
# the slope is in the first cell of the variable z. Python indexing begins at 0!!


# Add an annotation to the bottom left
textstr = f'Trend: {slope:.3f} °C/year'
plt.annotate(textstr, xy=(0.8, 0.05), xycoords='axes fraction',
             fontsize=12, verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

# Annotate the Warmest Year
#First get the year/value
warmest_year = int(yearly_avg.idxmax().values)
warmest_val = float(yearly_avg.max().values)

# Now add the annotation
plt.annotate(f'Warmest: {warmest_year}\n({warmest_val:.2f}°C)',
             xy=(warmest_year, warmest_val),
             xytext=(warmest_year - 15, warmest_val -0.5),
             arrowprops=dict(arrowstyle='->', lw=1.5, color='black'),
             fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8))

# Styling the plot
plt.title(f'ADD A TITLE', fontsize=15, pad=20)
plt.xlabel('ADD AN X AXIS LABEL', fontsize=12)
plt.ylabel('ADD A Y AXIS LABEL(°C)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

plt.tight_layout()

# Save the figure to your drive
save_name = 'Summer_temps_timeseries.png'
save_path = os.path.join(DATADIR, save_name)
plt.savefig(save_path, dpi = 300) # dpi = resolution of outut image

# Always call plt.show after it is saved
plt.show()



In [ ]:
#@title ####Predict the average temperature in 2050 and 2100


print(f"The predicted temperature in 2050 is {t2m_2050:.2f}")


### **Questions**



1.   What is the predicted temperature in 2050?
2.   What is the predicted temperature in 2100?
3.   What is the decadal warming trend?
4.   How reliable do you think a simple linear interpolation is?


### **TASK**
A visual inspection of our data suggests different trends at different times. From ~1945-1970 it looks like summer temperatures cool slightly before a continuous upward trend from 1970 onwards.

So, what happens to our predicted temperature in 2050 if we extrapolate the trend from 1970?

Try writing some code to do this....its not as hard as you might think.



In [ ]:
# @title #### Write some code to plot the temperature trend from 1970-2025 and then use this to predict the temperature in 2050

# Set up the figure
plt.figure(figsize=(12, 6))

# Plot the weighted summer means and set the line type etc
# xarray's .plot() wrapper automatically handles the time/year axis
yearly_avg.plot(marker='o', markersize=4, linestyle='-', color='#d62728', label='JJA Average')

# THIS IS WHERE YOU NEED TO ADD A LINE OF CODE TO SLICE THE YEARLY AVG TEMPS FROM 1970 - 2025
# Again using .sel and slice (by year)


# And then do the fitting exactly as before passing the new subset of data you made above
z_1970 =  # does the calculation for m and b
p_1970 =  # converts m and b into a convenient function.

# And plot this trendline
plt.plot(summer_1970.year, p_1970(summer_1970.year), color='black', linestyle='--', alpha=0.7, label='Linear Trend')

# Get the slope of the fitted line (e.g., how fast is it warming)
slope =  # the slope is in the first cell of the variable z. Remember Python indexing begins at 0!!

# Add an annotation to the bottom left
# xy=(----, ----) controls the location with respect to the axes
textstr = f'Post-1970 Trend: {slope:.3f} °C/year'
plt.annotate(textstr, xy=(0.7, 0.05), xycoords='axes fraction',
             fontsize=12, verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

# Annotate the Warmest Year
warmest_year = int(yearly_avg.idxmax().values)
warmest_val = float(yearly_avg.max().values)

plt.annotate(f'Warmest: {warmest_year}\n({warmest_val:.2f}°C)',
             xy=(warmest_year, warmest_val),
             xytext=(warmest_year - 15, warmest_val -0.5),
             arrowprops=dict(arrowstyle='->', lw=1.5, color='black'),
             fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8))

# Styling the plot
plt.title(f'Weighted Summer (JJA) Temperatures (ERA5): 1940 - 2025', fontsize=15, pad=20)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()




In [ ]:
#@title #### Use this cell to find the predicted temperatures in 2050 and 2100



print(f"The predicted temperature in 2050 is {t2m_2050:.2f}")

### **Questions**

1.  Using this new trend, what is the predicted temperatures in 2050 and 2100?
2.  How does this compare to the previous predictions?
3.  How could we assess if our trendlines are a good representation of our data?

## **Section 6: Calculate the anomaly between the warmest year and a baseline**



The plot above gives us some good information but it doesn't tell us anything about the spatial distribution of temperatures. Perhaps 2003 is dominated by exceptional warmth over a limited area? Let's visualise summer temperatures in Europe for 2003.

In [ ]:
#@title ##### This codecell plots a map of summer temps in 2003

# Set up the figure
plt.figure(figsize=(12, 8))

# Select the warmest year, we already have this stored as a variable so can use
# it here
t2m_2003 = summer_mean.sel(year=warmest_year)

# Define the projection (PlateCarree is standard for lat/lon data)
ax = plt.axes(projection=ccrs.PlateCarree())

# Plot the data
plot = t2m_2003.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap='plasma',       # You can try others by changing this
    cbar_kwargs={'shrink': 0.6, 'label': 'Temperature (°C)'}
)

# Add map features
ax.coastlines(resolution='50m', color='black', linewidth=1)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)

plt.title(f'Mean Summer (JJA) Temperature: {warmest_year}', fontsize=15)
plt.show()

So that shows us the absolute temperatures, but we want to know how unusual  these were compared to the average. Let's caluclate the anomaly between 2003 and a baseline of 1981-2010. In doing so we want to work with our unweigthed spatial data (`summer_mean`) as we are comparing each grid cell only to itself.

In effect we are asking, for every individual grid cell what is the difference between the 2003 summer temperature and the average summer temperature for the baseline period. The end result of this calculation is a 2-D map of the anomaly which we can easily visualise.

###**TASK**
Write a line of code to select all of the data in the years 1981-2010. Remember the handy `.sel` function and how we used it to select a slice of data in the previous workbook (across latitude and longitude that time).

Then calculate a mean across all of these years.

Then we need to write a line of code to calculate our anomaly. This is done simply by subtracting our baseline from our warmest year


In [ ]:
#@title ##### This codecell calculate the baseline and anomaly and makes a nice plot.

# Write some code to get data from 1981 - 2010 (the variable we are using is
# summer_mean where we have already averaged across the months for each year)

# Now we can calcualte a mean across all years (i.e., across the 'year' dimension)

# Calculate the anomaly


# Now lets plot it; Set up the figure
plt.figure(figsize=(12, 8))

# Define the projection (PlateCarree is standard for lat/lon data)
ax = plt.axes(projection=ccrs.PlateCarree())

# Our colormap would defualt to being divergent around the mean of our data (see
# plot above) but for an anomaly we want to make it divergent around zero. We
# can force it to do so by manually setting the limits for the scale and passing
# them into the plot function
v_limit = max(abs(anomaly.min()), abs(anomaly.max()))
plot = anomaly.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='RdBu_r',vmin=-v_limit, vmax=v_limit, cbar_kwargs={'shrink': 0.6, 'label': 'Temperature Anomaly (°C)'})

# Add map features
ax.coastlines(resolution='50m', color='black', linewidth=1)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.gridlines(draw_labels=True, linestyle='--', alpha=0.5)

plt.title(f'Summer (JJA) Temperature Anomaly: {warmest_year} - baseline (1981-2010)', fontsize=13)
plt.show()

In [ ]:
#@title #### You can write a line of code here to find the maximum anomaly

### **Questions**

1.  What is the maximum anomaly?   
2.  Why do we evalaute temperature anomalies rather than absolute values?
3.  Can you explain, in straight forward language, the calculations your 'anomaly calculate' code line performed?
4.  When mapping anomalies (which can be +ve, -ve, or zero), why is a diverging colourmap (e.g., coolwarm or RdBu_r) superior to a sequential colourmap (e.g., Viridis or YlOrRd)?

## **Summary and reflections**


In this notebook we have downloaded multi-decadal, multidimensional environmental data (air temperatures) and performed a simple analysis of a real (and historic) heatwave. The workflow follows a (hopefully) logical progression.

1.  Setting up our Google Colab environment with the packages/libraries we will need to perform our analysis.
2.  Mounting our Google Drive so the datasets (and outputs) can be used multiple times.
3.  Acquiring the data we require and importing it into Python in a useable format.
4. Inspecting the data; its dimensions and units so we are familar with it.
5. Reducing the 3-D data cube into 1-D or 2-D representations by slicing and aggregating the data along its various dimensions.
6. Performing statistical analyses (Trendline fitting and anomaly calculations) to gain insights and plotting the results.   

This notebook highlights why `xarray` is so useful. Instead of arranging large numbers of data points into clumsy multi-index tables, `xarray` treats dimensions (*Time, Lat, Lon*) as core metadata. This enables us to perform incredibly fast temporal/spatial selections (e.g., `.groupby('valid_time.year')`) and calculations while keeping the global location of every pixel intact. This highlights the value of self-describing data in environmental data science.

## **Extra activity**

What happens if you compare 2003 to a different baseline?

How much code would you need to change from the code box above to do so?